# DICOM Pixel Spacing 分析
讀取 `thyroid_old/data/CG_data/all_data` 中的 DICOM 檔，找出圖片 pixel 對應的真實大小（mm）

In [1]:
import pydicom
import os
import subprocess
import pandas as pd
from collections import defaultdict

In [2]:
DATA_ROOT = '/Users/tony.tu/Desktop/戴承智慧/thyroid_old/data/CG_data/all_data'

result = subprocess.run(['find', DATA_ROOT, '-name', '*.dcm'], capture_output=True, text=True)
all_files = result.stdout.strip().split('\n')
print(f'共找到 {len(all_files)} 個 dcm 檔')

共找到 39449 個 dcm 檔


In [3]:
# 從各個頂層子資料夾各取一個檔案（取樣）
dirs_seen = set()
selected = []
for f in all_files:
    parts = f.split('/')
    # all_data 底下第一層子資料夾
    idx = parts.index('all_data') + 1 if 'all_data' in parts else -1
    if idx >= 0 and idx < len(parts):
        top = parts[idx]
        if top not in dirs_seen:
            dirs_seen.add(top)
            selected.append(f)

print(f'從 {len(dirs_seen)} 個子資料夾各取 1 個，共 {len(selected)} 個檔案')
for f in selected:
    print(' ', '/'.join(f.split('/')[-4:]))

從 9 個子資料夾各取 1 個，共 9 個檔案
  11_4_data/FAF9E0AA82E42EB30279C72EF95A29B728201F5E/46651AB61E4B1108524A2A0430AA4C81BB7AECC4/1087d0a23c37432db48809f542c8c8a5.dcm
  12_16_data_normal/6F5B695A5FBAA874E763BD14FB4AFEE0299C86DB/D86652EB4669296E6EB59A061BD0F533BAE701EF/95639b2b5b31acefe0db2d84ab875a41.dcm
  10_31_data/2270F57D8121DCF7BBBB6121566624E37B60881B/971FC3241E1E79EB2A0AAD2633509ABAFB498E98/4fd8c49102c19cf88dabbeb5944fe746.dcm
  11_3_data/684C87E27194C4F5D888A4612CF04D65FEFB3B84/3030D7F94D9FE3C9DE3CBD949663192FF6D5AF66/6fa01f5a725c836a31b14132e9e01536.dcm
  10_21_data/78984C28DE98F48A09FF11E1DF66E91CAE1867B1/20221124/9ea7b2d705074a3ef80749103db11a9f.dcm
  1_5_normal/B548EA191D1DE8BAAD79CF0421B4F9D939654EC1/44FA46037797868CE0F1C9006758007112B2F588/0eaab3745e06312edfc8247e7dbba589.dcm
  12_26_data_nodule/1268387F084C766E5899EDF651940EAF4EB8E429/DF66390DA069EBA36FD5B2AB7D751C5EB8B4C4BF/1b3519309f3ec18b83742439b27793b0.dcm
  12_26_data_for_future/199B3A4486063702A2239D8C6BC6ABD953FBD644/A9183A5

In [4]:
def extract_pixel_info(filepath):
    """讀取 DICOM 並提取 pixel spacing 相關資訊"""
    ds = pydicom.dcmread(filepath, stop_before_pixels=True)
    
    rows = getattr(ds, 'Rows', None)
    cols = getattr(ds, 'Columns', None)
    modality = getattr(ds, 'Modality', 'N/A')
    
    info = {
        'file': '/'.join(filepath.split('/')[-4:]),
        'modality': modality,
        'rows': rows,
        'cols': cols,
        'spacing_source': None,
        'pixel_spacing_row_mm': None,
        'pixel_spacing_col_mm': None,
        'real_height_mm': None,
        'real_width_mm': None,
        'us_region_info': None,
    }
    
    if hasattr(ds, 'PixelSpacing'):
        ps = ds.PixelSpacing
        info['spacing_source'] = 'PixelSpacing'
        info['pixel_spacing_row_mm'] = float(ps[0])
        info['pixel_spacing_col_mm'] = float(ps[1])
        if rows and cols:
            info['real_height_mm'] = rows * float(ps[0])
            info['real_width_mm'] = cols * float(ps[1])
    
    elif hasattr(ds, 'ImagerPixelSpacing'):
        ps = ds.ImagerPixelSpacing
        info['spacing_source'] = 'ImagerPixelSpacing'
        info['pixel_spacing_row_mm'] = float(ps[0])
        info['pixel_spacing_col_mm'] = float(ps[1])
        if rows and cols:
            info['real_height_mm'] = rows * float(ps[0])
            info['real_width_mm'] = cols * float(ps[1])
    
    elif hasattr(ds, 'SequenceOfUltrasoundRegions'):
        # 超音波影像：用 SequenceOfUltrasoundRegions 取得物理尺寸
        regions = ds.SequenceOfUltrasoundRegions
        region_strs = []
        for i, reg in enumerate(regions):
            x0 = getattr(reg, 'RegionLocationMinX0', None)
            y0 = getattr(reg, 'RegionLocationMinY0', None)
            x1 = getattr(reg, 'RegionLocationMaxX1', None)
            y1 = getattr(reg, 'RegionLocationMaxY1', None)
            pdx = getattr(reg, 'PhysicalDeltaX', None)  # cm/px
            pdy = getattr(reg, 'PhysicalDeltaY', None)  # cm/px
            if pdx is not None and pdy is not None and None not in (x0, y0, x1, y1):
                w_mm = (int(x1) - int(x0)) * abs(float(pdx)) * 10
                h_mm = (int(y1) - int(y0)) * abs(float(pdy)) * 10
                dx_mm = abs(float(pdx)) * 10
                dy_mm = abs(float(pdy)) * 10
                region_strs.append(
                    f'Region{i}: ({x0},{y0})-({x1},{y1}), '
                    f'{dx_mm:.4f}x{dy_mm:.4f} mm/px, '
                    f'real={w_mm:.1f}x{h_mm:.1f} mm'
                )
                if i == 0:  # 用第一個 region 代表整體
                    info['spacing_source'] = 'UltrasoundRegions'
                    info['pixel_spacing_row_mm'] = dy_mm
                    info['pixel_spacing_col_mm'] = dx_mm
                    info['real_height_mm'] = h_mm
                    info['real_width_mm'] = w_mm
        info['us_region_info'] = ' | '.join(region_strs)
    
    return info

results = []
for f in selected:
    try:
        info = extract_pixel_info(f)
        results.append(info)
    except Exception as e:
        results.append({'file': '/'.join(f.split('/')[-4:]), 'error': str(e)})

print('讀取完成')

讀取完成


In [5]:
# 顯示結果
for r in results:
    print(f"{'='*60}")
    print(f"檔案: {r.get('file')}")
    if 'error' in r:
        print(f"  ❌ 錯誤: {r['error']}")
        continue
    print(f"  Modality     : {r['modality']}")
    print(f"  Image size   : {r['rows']} x {r['cols']} px")
    if r['spacing_source']:
        print(f"  Spacing來源  : {r['spacing_source']}")
        print(f"  Pixel Spacing: {r['pixel_spacing_row_mm']:.4f} mm/px (row)  x  {r['pixel_spacing_col_mm']:.4f} mm/px (col)")
        if r['real_height_mm']:
            print(f"  實際大小     : {r['real_height_mm']:.1f} mm (H)  x  {r['real_width_mm']:.1f} mm (W)")
        if r.get('us_region_info'):
            print(f"  US Regions   : {r['us_region_info']}")
    else:
        print(f"  ⚠️  找不到 pixel spacing 資訊")
print(f"{'='*60}")

檔案: 11_4_data/FAF9E0AA82E42EB30279C72EF95A29B728201F5E/46651AB61E4B1108524A2A0430AA4C81BB7AECC4/1087d0a23c37432db48809f542c8c8a5.dcm
  Modality     : US
  Image size   : 600 x 800 px
  Spacing來源  : UltrasoundRegions
  Pixel Spacing: 0.1630 mm/px (row)  x  0.1630 mm/px (col)
  實際大小     : 86.1 mm (H)  x  60.3 mm (W)
  US Regions   : Region0: (28,48)-(398,576), 0.1630x0.1630 mm/px, real=60.3x86.1 mm | Region1: (402,48)-(772,576), 0.1630x0.1630 mm/px, real=60.3x86.1 mm
檔案: 12_16_data_normal/6F5B695A5FBAA874E763BD14FB4AFEE0299C86DB/D86652EB4669296E6EB59A061BD0F533BAE701EF/95639b2b5b31acefe0db2d84ab875a41.dcm
  Modality     : US
  Image size   : 768 x 1024 px
  Spacing來源  : PixelSpacing
  Pixel Spacing: 0.0800 mm/px (row)  x  0.0800 mm/px (col)
  實際大小     : 61.4 mm (H)  x  81.9 mm (W)
檔案: 10_31_data/2270F57D8121DCF7BBBB6121566624E37B60881B/971FC3241E1E79EB2A0AAD2633509ABAFB498E98/4fd8c49102c19cf88dabbeb5944fe746.dcm
  Modality     : US
  Image size   : 768 x 1024 px
  Spacing來源  : Ultrasound

In [6]:
# 彙整成 DataFrame
df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'us_region_info'}
    for r in results if 'error' not in r
])

display_cols = ['file', 'modality', 'rows', 'cols', 'spacing_source',
                'pixel_spacing_row_mm', 'pixel_spacing_col_mm',
                'real_height_mm', 'real_width_mm']
df[display_cols].round(4)

,file,modality,rows,cols,spacing_source,pixel_spacing_row_mm,pixel_spacing_col_mm,real_height_mm,real_width_mm
0,11_4_data/FAF9E0AA82E42EB30279C72EF95A29B72820...,US,600,800,UltrasoundRegions,0.1630,0.1630,86.0627,60.3091
1,12_16_data_normal/6F5B695A5FBAA874E763BD14FB4A...,US,768,1024,PixelSpacing,0.0800,0.0800,61.4400,81.9200
2,10_31_data/2270F57D8121DCF7BBBB6121566624E37B6...,US,768,1024,UltrasoundRegions,0.0931,0.0931,50.0000,92.2719
3,11_3_data/684C87E27194C4F5D888A4612CF04D65FEFB...,US,600,800,UltrasoundRegions,0.1054,0.1054,47.8679,71.6963
4,10_21_data/78984C28DE98F48A09FF11E1DF66E91CAE1...,US,600,800,UltrasoundRegions,0.1251,0.1251,66.0627,46.2940
5,1_5_normal/B548EA191D1DE8BAAD79CF0421B4F9D9396...,US,600,800,UltrasoundRegions,0.1630,0.1630,86.0627,60.3091
6,12_26_data_nodule/1268387F084C766E5899EDF65194...,US,600,800,UltrasoundRegions,0.1054,0.1054,48.2896,71.6963
7,12_26_data_for_future/199B3A4486063702A2239D8C...,US,434,532,None,NaN,NaN,NaN,NaN
8,12_26_data_normal/CB616C4EB009DDDD642EE5B2A99E...,US,768,1024,PixelSpacing,0.0800,0.0800,61.4400,81.9200


In [7]:
# 統計各 modality 的 pixel spacing 分布（從全部檔案中取樣 200 個）
import random
random.seed(42)
sample_files = random.sample(all_files, min(200, len(all_files)))

sample_results = []
for f in sample_files:
    try:
        info = extract_pixel_info(f)
        sample_results.append(info)
    except:
        pass

df_sample = pd.DataFrame([r for r in sample_results if 'error' not in r and r['pixel_spacing_row_mm'] is not None])
print(f'有效樣本: {len(df_sample)}')
print()
print('各 modality 的 pixel spacing 統計 (mm/px):')
df_sample.groupby(['modality', 'spacing_source'])[['pixel_spacing_row_mm', 'pixel_spacing_col_mm']].agg(['mean', 'min', 'max']).round(4)

有效樣本: 172

各 modality 的 pixel spacing 統計 (mm/px):


pixel_spacing_row_mm                 \
                                           mean     min    max   
modality spacing_source                                          
US       PixelSpacing                    0.0764  0.0712  0.080   
         UltrasoundRegions               0.1402  0.0511  0.204   

                           pixel_spacing_col_mm                 
                                           mean     min    max  
modality spacing_source                                         
US       PixelSpacing                    0.0764  0.0712  0.080  
         UltrasoundRegions               0.1402  0.0511  0.204